In [36]:
import pandas as pd

In [37]:
dataset=pd.read_csv('mi_selected_features.csv')

In [38]:
def quanqual(dataset):
    quan=[]
    qual=[]
    for columnName in dataset.columns:
        if dataset[columnName].dtype=='O':
            qual.append(columnName)
        else:
            quan.append(columnName)
    return quan,qual
quan,qual=quanqual(dataset)

In [39]:
descriptive=pd.DataFrame(index=["Mean","Median","Mode","Max","Min","25%","50%","75%","100%","IQR","1.5IQR","LesserRange","GreaterRange","Skew","Kurtosis"],columns=quan)
for columnName in quan:
    descriptive.loc["Mean",columnName]=dataset[columnName].mean()
    descriptive.loc["Median",columnName]=dataset[columnName].median()
    descriptive.loc["Mode",columnName]=dataset[columnName].mode()[0]
    descriptive.loc["Max",columnName]=dataset[columnName].max()
    descriptive.loc["Min",columnName]=dataset[columnName].min()

    descriptive.loc["25%",columnName]=dataset.describe()[columnName]["25%"]
    descriptive.loc["50%",columnName]=dataset.describe()[columnName]["50%"]
    descriptive.loc["75%",columnName]=dataset.describe()[columnName]["75%"]
    descriptive.loc["100%",columnName]=dataset.describe()[columnName]["max"]

    descriptive.loc["IQR",columnName]=descriptive.loc["75%",columnName]-descriptive.loc["25%",columnName]
    descriptive.loc["1.5IQR",columnName]=1.5*descriptive.loc["IQR",columnName]
    descriptive.loc["LesserRange",columnName]=descriptive.loc["25%",columnName]-descriptive.loc["1.5IQR",columnName]
    descriptive.loc["GreaterRange",columnName]=descriptive.loc["75%",columnName]+descriptive.loc["1.5IQR",columnName]
    descriptive.loc["Skew",columnName]=dataset[columnName].skew()
    descriptive.loc["Kurtosis",columnName]=dataset[columnName].kurt()

In [40]:
for columnName in quan:
    if columnName!='LET_IS':
        lower=descriptive.loc["LesserRange",columnName]
        upper=descriptive.loc["GreaterRange",columnName]
        dataset[columnName]=dataset[columnName].clip(lower=lower,upper=upper)

In [41]:
dataset['target'] = (dataset['LET_IS'] > 0).astype(int)

In [42]:
indep=dataset.drop(['LET_IS','target'],axis=1)
dep=dataset['target']

In [43]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(indep,dep,test_size=1/3,random_state=42,stratify=dep)

In [44]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train=sc.fit_transform(X_train)
X_test=sc.transform(X_test)

In [45]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

In [46]:
from sklearn.ensemble import RandomForestClassifier
Classifier = RandomForestClassifier(n_estimators=300,criterion='entropy',class_weight='balanced',random_state=42)
Classifier.fit(X_train_sm, y_train_sm) 

RandomForestClassifier(class_weight='balanced', criterion='entropy',
                       n_estimators=300, random_state=42)

In [47]:
y_pred=classifier.predict(X_test)

C:\Users\SS\anaconda3\envs\aiml\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but SVC was fitted with feature names
  warnings.warn(


In [48]:
from sklearn.metrics import confusion_matrix
cm=confusion_matrix(y_test,y_pred)

In [49]:
from sklearn.metrics import classification_report
clf_report=classification_report(y_test,y_pred)
print(clf_report)

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       477
           1       0.16      1.00      0.27        90

    accuracy                           0.16       567
   macro avg       0.08      0.50      0.14       567
weighted avg       0.03      0.16      0.04       567



C:\Users\SS\anaconda3\envs\aiml\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\SS\anaconda3\envs\aiml\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\SS\anaconda3\envs\aiml\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [34]:
cm

array([[  0, 477],
       [  0,  90]], dtype=int64)

In [35]:
from sklearn.metrics import roc_auc_score

probs = classifier.predict_proba(X_test)[:,1]
print("ROC-AUC:", roc_auc_score(y_test, probs))

ROC-AUC: 0.7773119030980666


C:\Users\SS\anaconda3\envs\aiml\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but SVC was fitted with feature names
  warnings.warn(
